In [ ]:
#Qué hace: agrupa variables fisiológicas en dominios clínicos funcionales.
#Clave: estructura clínica e interpretabilidad (inspiración en la literatura, no framework cerrado).

In [1]:
# Imports y carga de datos

In [2]:
import pandas as pd
import numpy as np

df = pd.read_parquet("06_variables_diarias.parquet")

# Orden imprescindible para tendencias
df = df.sort_values(["subject_id", "hadm_id", "icu_stay_id", "day_idx"]).reset_index(drop=True)

# Tipos recomendados
df["day_idx"] = df["day_idx"].astype(int)
df["icu_stay_id"] = df["icu_stay_id"].astype("Int64")

In [3]:
def normalize_fio2(x):
    """
    Convert FiO2 to fraction.
    - If x is 21–100 -> assume percent and divide by 100
    - If x is 0–1.5 -> assume already fraction
    - Otherwise -> NaN
    """
    if pd.isna(x):
        return np.nan
    try:
        x = float(x)
    except Exception:
        return np.nan
    
    if 1.5 < x <= 100:      # e.g., 25, 40, 60
        return x / 100.0
    if 0 < x <= 1.5:        # e.g., 0.25, 0.4
        return x
    return np.nan

df["FiO2_frac"] = df["FiO2"].apply(normalize_fio2)

# Recalcula S/F (no uses el SF_ratio previo)
df["SF_ratio_clean"] = df["SpO2"] / df["FiO2_frac"]

In [4]:
def temp_ok(x):
    if pd.isna(x):
        return np.nan
    return 1 if 36 <= x <= 38 else 0

def map_ok(x):
    if pd.isna(x):
        return np.nan
    return 1 if x >= 65 else 0

def lactate_low(x):
    if pd.isna(x):
        return np.nan
    return 1 if x < 2 else 0

def sf_ok(x):
    if pd.isna(x):
        return np.nan
    return 1 if x >= 240 else 0

df["temp_ok"] = df["Temp"].apply(temp_ok)
df["map_ok"]  = df["MAP"].apply(map_ok)
# Si lactato no está en tu 06, esto no se ejecuta: solo aplícalo cuando exista esa columna
# df["lactate_low"] = df["Lactate"].apply(lactate_low)

df["sf_ok"] = df["SF_ratio_clean"].apply(sf_ok)

In [5]:
def wbc_normalizing(series):
    normal_low, normal_high = 4, 12
    s = series.astype("float")
    dist = (s - s.clip(normal_low, normal_high)).abs()
    d = dist.diff()
    out = pd.Series(np.nan, index=series.index)
    mask = d.notna()
    out.loc[mask & (d < 0)] = 1
    out.loc[mask & (d == 0)] = 0
    out.loc[mask & (d > 0)] = -1
    return out

# Solo si existe la columna:
# df["wbc_trend"] = (
#     df.groupby(["subject_id","hadm_id","icu_stay_id"])["WBC"]
#       .apply(wbc_normalizing)
#       .reset_index(level=[0,1,2], drop=True)
# )

In [6]:
out_cols = [
    "subject_id","hadm_id","icu_stay_id","day_idx",
    "Temp","MAP","SpO2","FiO2","FiO2_frac",
    "SF_ratio_clean",
    "temp_ok","map_ok","sf_ok"
]

df_07 = df[out_cols].copy()
df_07.to_parquet("07_dominios_clinicos.parquet", index=False)

df_07.head()

,subject_id,hadm_id,icu_stay_id,day_idx,Temp,MAP,SpO2,FiO2,FiO2_frac,SF_ratio_clean,temp_ok,map_ok,sf_ok
0,10001217,24597018,37067082,0,36.972222,90.0,98.0,NaN,NaN,NaN,1.0,1.0,NaN
1,10001217,24597018,37067082,1,37.611111,94.0,95.0,NaN,NaN,NaN,1.0,1.0,NaN
2,10002428,20321825,34807493,0,36.944444,63.0,100.0,25.0,0.25,400.0,1.0,0.0,1.0
3,10002428,20321825,34807493,1,37.027778,67.5,97.0,NaN,NaN,NaN,1.0,1.0,NaN
4,10002428,23473524,35479615,0,36.666667,82.0,99.0,NaN,NaN,NaN,1.0,1.0,NaN
